# Exercise 2 – Use Case 4: RAG Chatbot with Wikipedia

---

## Report

### Chosen Models
| Role | Model | Why | Link |
|------|-------|-----|----|
| **Retriever** | `sentence-transformers/all-MiniLM-L6-v2` | Fast, small (80MB), great for semantic search | https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2 |
| **Generator** | `google/flan-t5-base` | Instruction-following, small (250MB) | https://huggingface.co/google/flan-t5-base |

### Dataset
- **Wikipedia** (HuggingFace `datasets`), English, `20220301.en` split  
- We use **5,000 passages** (one per article intro)  
- Each passage = article title + first ~500 chars of text

### Key Hyperparameters
- `top_k = 3` → how many Wikipedia passages to retrieve
- `max_new_tokens = 200` → max answer length
- `temperature = 0.0`

### Challenges & Solutions
- **Memory** → used base/small models only; 5k passages not 100k
- **Speed** → precomputed embeddings once; cosine search is instant
- **Quality** → flan-t5 works better than plain t5 because it's instruction-tuned

### Results Preview
We compare answers *with retrieval* vs *without retrieval* in Section 6.  
RAG answers are noticeably more factual and specific.

---

## Install & Import

In [1]:
!pip install -q sentence-transformers datasets transformers accelerate

In [2]:
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

## Load Wikipedia Dataset
We grab 5,000 article introductions from Wikipedia.  
Each one becomes a *passage* the chatbot can retrieve from.

In [3]:
# Load only 5000 articles from Wikipedia
NUM_PASSAGES = 5000

print(f"Loading {NUM_PASSAGES} Wikipedia articles...")

wiki_stream = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True
)

# Collect passages: title + first 500 characters of text
passages = []
for i, article in enumerate(wiki_stream):
    if i >= NUM_PASSAGES:
        break
    title = article["title"]
    text = article["text"][:500].strip()
    passages.append(f"{title}: {text}")

print(f"Loaded {len(passages)} passages")

Loading 5000 Wikipedia articles...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loaded 5000 passages


## Build the Retrieval Index
We encode every passage into a **vector**.  
Similar texts → similar vectors → we can find relevant passages by comparing them.

This is called **semantic search** (understands meaning, not just keywords).

In [4]:
# Load the retriever model
print("Loading retriever model...")
retriever = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Encode all 5000 passages into vectors
# batch_size=64 means we process 64 passages at a time → faster
print("Encoding passages...")
passage_embeddings = retriever.encode(
    passages,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nIndex ready! Shape: {passage_embeddings.shape}")

Loading retriever model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding passages...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]


Index ready! Shape: (5000, 384)


## Load the Generator Model
`flan-t5-base` is an instruction-tuned model.

In [5]:
print("Loading generator model (flan-t5-base)...")

model_name = "google/flan-t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def generate(prompt, max_new_tokens=200, temperature=0.0):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=temperature if temperature > 0 else None,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Generator ready! (running on {device})")

Loading generator model (flan-t5-base)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator ready! (running on cuda)


## The RAG Pipeline
Two helper functions:
1. `retrieve(query, top_k)` → find the most relevant Wikipedia passages
2. `rag_answer(question, top_k, max_new_tokens, temperature)` → full RAG answer

**Cosine similarity** = how aligned two vectors are (1.0 = identical, 0 = unrelated)

In [6]:
def retrieve(query, top_k=3):
    """
    Given a question, find the top_k most relevant Wikipedia passages.
    Returns list of (score, passage_text).
    """
    # Encode the question into a vector
    query_vec = retriever.encode([query], convert_to_numpy=True)   # shape (1, 384)

    # Cosine similarity: dot product after normalizing
    scores = passage_embeddings @ query_vec.T
    scores = scores.flatten()

    # Pick top_k highest scores
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = [(float(scores[i]), passages[i]) for i in top_indices]
    return results


def rag_answer(question, top_k=3, max_new_tokens=200, temperature=0.0):
    """
    Full RAG pipeline:
    1. Retrieve top_k relevant passages
    2. Build a prompt: context + question
    3. Generate an answer
    """
    # Step 1: Retrieve
    retrieved = retrieve(question, top_k=top_k)
    context = "\n\n".join([passage for _, passage in retrieved])

    # Step 2: Build prompt for flan-t5
    prompt = (
        f"Answer the question based on the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        f"Answer:"
    )

    # Step 3: Generate
    answer = generate(prompt, max_new_tokens=max_new_tokens, temperature=temperature)
    return answer, retrieved


def no_rag_answer(question, max_new_tokens=200):
    """Answer WITHOUT retrieval — just the raw model with no context."""
    return generate(f"Question: {question}\nAnswer:", max_new_tokens=max_new_tokens, temperature=0)

## Compare: With RAG vs Without RAG
This is the analysis part. We ask the same questions both ways and compare.  

In [7]:
test_questions = [
    "Who was Albert Einstein?",
    "What is machine learning?",
]

for q in test_questions:
    print("=" * 60)
    print(f"QUESTION: {q}")
    print()

    # Without RAG
    plain = no_rag_answer(q)
    print(f"WITHOUT RAG:\n   {plain}")
    print()

    # With RAG
    rag, retrieved_docs = rag_answer(q, top_k=3)
    print(f"WITH RAG:\n   {rag}")
    print()

    # Show which Wikipedia passages were retrieved
    print(f"Retrieved passages (scores):")
    for score, passage in retrieved_docs:
        print(f"   [{score:.3f}] {passage[:100]}...")
    print()

QUESTION: Who was Albert Einstein?

WITHOUT RAG:
   physicist

WITH RAG:
   a German-born theoretical physicist

Retrieved passages (scores):
   [0.734] Albert Einstein: Albert Einstein ( ; ; 14 March 1879 – 18 April 1955) was a German-born theoretical ...
   [0.533] Albertus Magnus: Albertus Magnus  ( – 15 November 1280), also known as Saint Albert the Great or Alb...
   [0.526] Albert II: Albert II may refer to:

Monkeys 
 Albert II (monkey), first primate and first mammal in ...

QUESTION: What is machine learning?

WITHOUT RAG:
   Machine learning is a technology that enables people to learn from data.

WITH RAG:
   field of study in computer science that develops and studies intelligent machines

Retrieved passages (scores):
   [0.542] Artificial intelligence: Artificial intelligence (AI) is the intelligence of machines or software, a...
   [0.484] Algorithm: In mathematics and computer science, an algorithm () is a finite sequence of rigorous ins...
   [0.466] Bioinformatics: Bio

## Experiment with Parameters

In [8]:
question = "What are the effects of climate change?"

for top_k in [1, 3, 5]:
    answer, _ = rag_answer(question, top_k=top_k, max_new_tokens=150, temperature=0.0)
    print(f"top_k={top_k}: {answer}")
    print()

top_k=1: greenhouse gases

top_k=3: greenhouse gases

top_k=5: Antarctica



In [9]:
question = "What is quantum physics?"

for temp in [0.0, 0.5, 1.0]:
    answer, _ = rag_answer(question, top_k=3, max_new_tokens=150, temperature=temp)
    print(f"temperature={temp}: {answer}")
    print()

temperature=0.0: field of physics that studies atoms as an isolated system of electrons and an atomic nucleus

temperature=0.5: Ein and protocatman the fusion reactions used around them have never developed, with nothing for this explanation compared except experimental. But quantum computing (intomical energy exchange

temperature=1.0: to posses the interlvehenral resulting chemical reaction where some neutral charged, some pro and reactive posiitvo and others do nothing (by means and act etcassments is also explained here)), without significant impact



## Step 8 – Interactive Chatbot (CLI)
Type `quit` to stop.

In [10]:
print("=" * 50)
print(" RAG Chatbot – powered by Wikipedia")
print(" Type 'quit' to exit")
print("=" * 50)
print()

# Chatbot config — you can change these
TOP_K       = 3
MAX_TOKENS  = 200
TEMPERATURE = 0.0

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue
    if user_input.lower() in ["quit", "exit", "bye"]:
        print("Bot: Goodbye!")
        break

    # Generate answer with RAG
    answer, retrieved_docs = rag_answer(
        user_input,
        top_k=TOP_K,
        max_new_tokens=MAX_TOKENS,
        temperature=TEMPERATURE
    )

    print(f"\nBot: {answer}")
    print()

    # Optionally show which passages were used
    show_sources = input("Show sources? (y/n): ").strip().lower()
    if show_sources == "y":
        print("\n Retrieved from Wikipedia:")
        for i, (score, passage) in enumerate(retrieved_docs, 1):
            print(f"  [{i}] score={score:.3f} | {passage[:120]}...")
    print()

 RAG Chatbot – powered by Wikipedia
 Type 'quit' to exit

You: What is AI?

Bot: the intelligence of machines or software

Show sources? (y/n): y

 Retrieved from Wikipedia:
  [1] score=0.781 | Artificial intelligence: Artificial intelligence (AI) is the intelligence of machines or software, as opposed to the int...
  [2] score=0.741 | Ai: AI is artificial intelligence, intellectual ability in machines and robots.

Ai, AI or A.I. may also refer to:

Anim...
  [3] score=0.602 | AI-complete: In the field of artificial intelligence, the most difficult problems are informally known as AI-complete or...

You: What is coding?

Bot: a system of rules to convert information

Show sources? (y/n): y

 Retrieved from Wikipedia:
  [1] score=0.661 | Code: In communications and information processing, code is a system of rules to convert information—such as a letter, w...
  [2] score=0.634 | Computer programming: Computer programming or coding is the composition of sequences of instructions, called 